# Ch.2 データ可視化・探索的データ分析（EDA）

## 本チャプターのゴール
- グラフを使ってデータの分布・傾向を視覚的に把握できる
- 相関分析でデータ間の関係を見つけられる
- グラフから「仮説」を立てる思考を身につける

## EDA（Exploratory Data Analysis）とは
モデルを作る前に、データの**特徴・傾向・外れ値・相関**を可視化によって把握する作業です。
データサイエンティストの実務時間の**50〜70%**はこのEDAが占めると言われています。

---

## 🤖 生成AIの使い方ガイド（本チャプター）

| AIに任せてよいこと | 自分で理解すること |
|---|---|
| グラフのサイズ・色・フォント設定 | このグラフから何が読み取れるか |
| `seaborn.heatmap` の引数 | 相関が高い変数の組み合わせがなぜ意味を持つか |
| `subplot` でグラフを並べる方法 | 可視化の目的（何を確認したいか）の設定 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # 日本語フォント対応
from sklearn.datasets import load_wine

# データ準備（Ch.1と同じ）
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('データ準備完了:', df.shape)

---
## 2.1 ヒストグラム：データの分布を見る

In [ ]:
# アルコール度数の分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 全体の分布
axes[0].hist(df['alcohol'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('アルコール度数の分布（全体）')
axes[0].set_xlabel('アルコール度数')
axes[0].set_ylabel('件数')

# 品種別の分布
for name, group in df.groupby('品種'):
    axes[1].hist(group['alcohol'], bins=15, alpha=0.6, label=name, edgecolor='white')
axes[1].set_title('アルコール度数の分布（品種別）')
axes[1].set_xlabel('アルコール度数')
axes[1].set_ylabel('件数')
axes[1].legend()

plt.tight_layout()
plt.show()

# 観察：Barolo は全体的にアルコール度数が高い傾向がある

---
## 2.2 箱ひげ図：品種間の分布を比較する

In [ ]:
# 品種ごとの色の濃さ（color_intensity）を比較
fig, ax = plt.subplots(figsize=(8, 5))

df.boxplot(column='color_intensity', by='品種', ax=ax,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))

ax.set_title('品種ごとの色の濃さ（color_intensity）')
ax.set_xlabel('品種')
ax.set_ylabel('color_intensity')
plt.suptitle('')  # デフォルトのタイトルを消す
plt.tight_layout()
plt.show()

# 観察：Barolo は色が特に濃く、Grignolino は淡い

---
## 2.3 散布図：2変数の関係を見る

In [ ]:
# アルコール度数 vs プロリン含有量（品種で色分け）
colors = {'Barolo': '#e74c3c', 'Grignolino': '#2ecc71', 'Barbera': '#3498db'}

fig, ax = plt.subplots(figsize=(8, 6))
for name, group in df.groupby('品種'):
    ax.scatter(group['alcohol'], group['proline'],
               label=name, color=colors[name], alpha=0.7, s=60)

ax.set_title('アルコール度数 vs プロリン含有量')
ax.set_xlabel('アルコール度数')
ax.set_ylabel('プロリン含有量 (proline)')
ax.legend(title='品種')
plt.tight_layout()
plt.show()

# 観察：品種によってクラスターが形成されている → 分類に使えそうな特徴量

---
## 2.4 相関ヒートマップ：変数間の相関を一覧で見る

In [ ]:
# 相関行列の計算
numeric_cols = wine.feature_names  # 数値列のみ
corr_matrix = df[numeric_cols].corr()

# ヒートマップで表示
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix,
            annot=True,       # 数値を表示
            fmt='.2f',        # 小数点2桁
            cmap='coolwarm',  # 赤→青のカラーマップ
            center=0,
            square=True,
            ax=ax)
ax.set_title('特徴量間の相関係数ヒートマップ')
plt.tight_layout()
plt.show()

# 読み方：1に近いほど正の相関、-1に近いほど負の相関

In [ ]:
# 相関が強い上位の組み合わせを確認
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs < 1.0]  # 自己相関を除く
corr_pairs = corr_pairs.abs().sort_values(ascending=False)
print('相関が強い変数の組み合わせ（上位5）:')
print(corr_pairs.head(10)[::2])  # 重複を除いて上位5組

---
## 2.5 seaborn の pairplot：特徴量の組み合わせを一括確認

In [ ]:
# 代表的な4特徴量で pairplot（全組み合わせの散布図）
selected_cols = ['alcohol', 'color_intensity', 'proline', 'flavanoids', '品種']

sns.pairplot(df[selected_cols], hue='品種',
             diag_kind='hist',
             plot_kws={'alpha': 0.6})
plt.suptitle('主要特徴量のペアプロット', y=1.02)
plt.show()

# 観察：品種が視覚的に分離できる特徴量の組み合わせはどれか？

---
## 🖊️ 演習（40分）

**ヒント：** グラフ生成のコードはAIに任せてOKです。「何を見たいか」を決めてプロンプトを書き、生成されたコードを実行した上で、結果の「解釈」を自分の言葉で書いてください。

---

### 問1：棒グラフで品種ごとの平均アルコール度数を比較する

In [ ]:
# TODO: 品種ごとの平均アルコール度数を棒グラフで表示してください
# ヒント：groupby → mean → plot(kind='bar') または plt.bar() を使います



### 問2：自分で選んだ2つの特徴量の散布図を作る

In [ ]:
# TODO: 相関ヒートマップを参考に、相関が高そうな2変数を選んで
#       品種で色分けした散布図を作成してください
#
# 選んだ変数1: ____________
# 選んだ変数2: ____________
# 選んだ理由:  ____________



### 問3：グラフから「仮説」を1つ言語化する

作ったグラフを見て、気づいたことをまとめてください。

**気づいたこと（自由記述）：**

（例）「Barolo はアルコール度数が高く、プロリンも多い傾向がある。この2つの特徴量だけで、Barolo と他の品種をかなり区別できそうだ。」

→ あなたの仮説：




### 問4（発展）：箱ひげ図を複数特徴量で並べて比較する

In [ ]:
# TODO: 'alcohol', 'color_intensity', 'proline', 'flavanoids' の4つを
#       subplot(2,2) で並べて箱ひげ図を表示してください
# ヒント：plt.subplots(2, 2) → 各 ax に boxplot を描く



---
## ✅ チャプターのまとめ

| グラフ | 目的 | コード |
|--------|------|--------|
| ヒストグラム | 分布の形を見る | `plt.hist()` |
| 箱ひげ図 | グループ間の分布を比較 | `df.boxplot()` or `sns.boxplot()` |
| 散布図 | 2変数の関係を見る | `plt.scatter()` |
| ヒートマップ | 相関を一覧で見る | `sns.heatmap(df.corr())` |
| ペアプロット | 全変数ペアを一括確認 | `sns.pairplot()` |

**EDAの進め方（実務でも使えるフロー）**
1. 分布を確認（ヒストグラム）
2. グループ間を比較（箱ひげ図）
3. 変数間の関係を確認（散布図・ヒートマップ）
4. 気づきを言語化して「仮説」にする